# ☁️ Notebook 03: Gestión de Versiones y Persistencia en S3 (MinIO)

La finalidad de este notebook es asegurar que el modelo optimizado de **Sentinel** sea almacenado de forma segura y organizada en nuestro almacenamiento de objetos (**MinIO**). Este paso garantiza la persistencia a largo plazo y la trazabilidad de los modelos generados por el pipeline de **MLOps**.

---

### 🎯 Objetivos Funcionales

#### 1. Centralización del Modelo
Mueve el artefacto `model.onnx` y sus archivos de configuración desde el sistema de archivos efímero del nodo de entrenamiento hacia un **Bucket de S3** centralizado. Esto permite que cualquier servicio del clúster (como el Model Server) pueda acceder al modelo.

#### 2. Lógica de Versionado Incremental (v1, v2, v3...)
Para evitar la sobreescritura accidental y permitir estrategias de **Rollback** (volver a una versión anterior), el notebook implementa una lógica inteligente:
* Escanea el contenido actual del Bucket.
* Detecta la última versión existente (ej: `v5`).
* Crea automáticamente una nueva ruta jerárquica para la nueva versión (ej: `v6/`).

#### 3. Integración con el Data Connection de OpenShift AI
El notebook utiliza las credenciales de servicio (Access Key y Secret Key) configuradas en el entorno de **OpenShift AI** para autenticarse con MinIO, garantizando que el proceso de subida sea seguro y auditado.

---

### 🏗️ Flujo de Operación
1. **Conexión:** Se establece el vínculo con el servidor MinIO de Movistar.
2. **Identificación:** Se determina el prefijo de versión correspondiente.
3. **Transferencia:** Se suben los archivos críticos (`model.onnx`, `tokenizer.json`, `config.json`).
4. **Validación:** Se confirma que el tamaño del archivo en S3 coincida con el local para asegurar una carga íntegra.

---

> **Resultado Final:** Al terminar este notebook, el modelo queda disponible en una URL persistente, lista para ser consumida por el recurso **InferenceService** en la etapa de despliegue.

# 📦 Persistencia con Versionado Automático en MinIO (S3)

En esta sección, implementamos la lógica de **Model Registry** simplificada. El objetivo es mover los artefactos optimizados desde el entorno de ejecución local hacia un almacenamiento de objetos persistente, asegurando que cada ejecución del pipeline genere una nueva versión sin sobreescribir las anteriores.

---

### 🔧 1. Configuración de Conectividad
Establecemos la conexión con el servicio de **MinIO** interno del clúster de OpenShift. 
* **Endpoint:** Utilizamos el DNS interno del servicio para minimizar la latencia.
* **Addressing Style:** Configuramos el acceso tipo `path` para compatibilidad con implementaciones locales de S3.

### 🔢 2. Lógica de Versionado Incremental (`v+1`)
Para automatizar el despliegue, el código realiza las siguientes acciones:
1. **Inspección:** Escanea el bucket `modelos-prod` buscando prefijos que sigan el patrón `v*` (ej: `v1/`, `v2/`).
2. **Extracción:** Utiliza **Expresiones Regulares (Regex)** para identificar el número de versión más alto.
3. **Cálculo:** Incrementa el contador para definir la ruta de la nueva versión (ej: si existe `v3`, la nueva será `v4`).

### 🚀 3. Transferencia de Artefactos
Se realiza una subida iterativa de todos los archivos contenidos en `./modelo_t-moviles_onnx`. Esto incluye:
* El grafo del modelo (`model.onnx`).
* Las configuraciones del modelo y el tokenizer (`config.json`, `tokenizer.json`, etc.).

---
> **Resultado esperado:** Una vez finalizada la ejecución, el modelo estará disponible en una ruta jerárquica dentro de MinIO

In [ ]:
import os
import boto3
import re
from botocore.client import Config

endpoint = "http://minio-service.t-moviles-ai.svc.cluster.local:9000"
creds = {"key": "minio", "secret": "minio123"}
bucket_name = "modelos-prod"
local_folder = "./modelo_t-moviles_onnx"

s3 = boto3.client('s3', endpoint_url=endpoint, 
                  aws_access_key_id=creds["key"], 
                  aws_secret_access_key=creds["secret"],
                  config=Config(signature_version='s3v4', s3={'addressing_style': 'path'}),
                  region_name='none')

def get_latest_version_number(bucket):
    """Escanea el bucket buscando carpetas con formato 'vX'"""
    try:
        response = s3.list_objects_v2(Bucket=bucket, Delimiter='/')
        prefixes = response.get('CommonPrefixes', [])
        
        versions = [0] #Por si el bucket está vacío
        for p in prefixes:
            folder = p.get('Prefix').strip('/')
            # Extraemos el número de carpetas tipo 'v1', 'v2', etc.
            match = re.match(r'^v(\d+)$', folder)
            if match:
                versions.append(int(match.group(1)))
        
        return max(versions)
    except Exception as e:
        print(f"⚠️ No se pudo leer versiones previas (asumiendo v0): {e}")
        return 0

def upload_incrementing_version():
    if not os.path.exists(local_folder) or not os.listdir(local_folder):
        print(f"❌ Error: No hay archivos en {local_folder} para subir.")
        return

    current_max = get_latest_version_number(bucket_name)
    new_version = f"v{current_max + 1}"
    
    print(f"🔍 Última versión detectada: v{current_max}")
    print(f"🚀 Subiendo nueva versión: {new_version}...")

    for file in os.listdir(local_folder):
        local_path = os.path.join(local_folder, file)
        if os.path.isfile(local_path):
            s3_key = f"{new_version}/{file}"
            s3.upload_file(local_path, bucket_name, s3_key)
            print(f"   ✅ {s3_key}")

    print(f"\n✨ ¡Listo! Modelo T-Moviles Argentina {new_version} desplegado en MinIO.")
    return new_version

version_desplegada = upload_incrementing_version()